# Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [32]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,AIMessage

## Message-based Summarization
agent = create_agent(
    model = "groq:qwen/qwen3.8-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.8-27b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [16]:
### Run with Thread id
config = {"configurable":{"thread_id":"test-1"}}

In [17]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content="q")]},config)
    print(f"Messages:{response}")
    print(f"Messages:{len(response['messages'])}")

Messages:{'messages': [HumanMessage(content='q', additional_kwargs={}, response_metadata={}, id='d936048b-b80f-4323-aa2a-87de3d2028fe'), AIMessage(content='It looks like you might have sent a message by mistake or are testing the connection. How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 13, 'total_tokens': 38, 'completion_time': 0.048356227, 'completion_tokens_details': None, 'prompt_time': 0.000801518, 'prompt_tokens_details': None, 'queue_time': 0.066796112, 'total_time': 0.049157745}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0f2d9-7a5e-71f2-af7e-56c13db76251-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 25, 'total_tokens': 38})]}
Messages:2
Messages:{'messages': [HumanMessage(content='q', additional_kwargs={}, r

## Token Size

In [19]:
from groq.types import model
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.8-27b",
            trigger=("tokens",400),
            keep=("tokens",200),
        ),
    ]
)
config={"configurable":{"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars= sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [20]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke({"messages":[HumanMessage(content=f"Find hotel in {city}")]}
    ,config = config)

tokens = count_tokens(response["messages"])
print(f"{city} : ~{tokens}, {len(response['messages'])} : messages")
print(f"{response['messages']}")

Singapore : ~556, 6 : messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nFind hotel accommodations in various cities. The current active request is for hotels in Dubai, following previous searches for Tokyo, London, Paris, and New York where no bookings were made.\n\n## SUMMARY\n- **Current Status**: Dubai hotel options have been successfully retrieved via tool call and are ready to be presented to the user.\n- **Dubai Options Retrieved**:\n  1. **Grand Hotel**: 5-star, $350/night (Amenities: spa, pool, gym).\n  2. **City Inn**: 4-star, $180/night (Amenities: business center).\n  3. **Budget Stay**: 3-star, $75/night (Amenities: free wifi).\n- **Previous Context**:\n  - **Tokyo, London, Paris, New York**: Searches were previously conducted in these cities. Identical hotel names, pricing, and amenities were presented for each. No selections or bookings were made for any of these cities.\n- **User Interaction**: The user has not yet provi

## Human In the Loop MiddleWare
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [23]:
agent=create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","reject","edit"],
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [25]:
config={"configurable":{"thread_id":"test-approve"}}
# Step-1: Request
result = agent.invoke({
    "messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]
    },config = config)

In [26]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='3750872e-7429-404a-9f3f-e39b252fa3a6'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'gtxzge18m', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 376, 'total_tokens': 433, 'completion_time': 0.178034174, 'completion_tokens_details': None, 'prompt_time': 0.026664997, 'prompt_tokens_details': None, 'queue_time': 0.051288761, 'total_time': 0.204699171}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0f2ef-1c67-7920-b562-8bf7150f52f3-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': '

In [28]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config = config
    )
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: Email sent to john@test.com with subject 'Hello' and body 'How are you?'.


### Reject

In [34]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [35]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [36]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: It looks like the email to john@test.com was not sent — the action was rejected. Let me know if you'd like me to try again or make any changes to the message.


### Editing

In [37]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [38]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [39]:
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[{
                    "type":"edit",
                    "edited_action":{
                        "name":"send_email_tool",
                        "args":{
                            "recipient":"correct@email.com",
                            "subject": "Corrected Subject",
                            "body": "This was edited by human before sending"
                        }
                    }
                }]
            }
        ),config=config
    )

    print(f"Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
Result: The email was sent, but a human reviewer modified it before it went out. It was actually sent to **correct@email.com** with the subject **"Corrected Subject"** and a different body, rather than the original details you provided.

Let me know if you'd like me to resend the email with your original content.
